# LLaMA-2-13B FP16 Zero-Shot Accuracy Baseline

Evaluate the original FP16 `meta-llama/Llama-2-13b-hf` model with the same six zero-shot tasks reported by BiE: LAMBADA, ARC-Easy, PIQA, COPA, QNLI, and SST-2.

The notebook uses `lm-evaluation-harness==0.4.13`, evaluates every example (`limit=None`), checks the expected sample count of every task, and stores both the complete harness output and a compact BiE-compatible summary. No quantization is applied.

In [ ]:
%pip install -q "lm_eval[hf]==0.4.13" sentencepiece

In [ ]:
import importlib.metadata
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import datasets
import lm_eval
import torch
import transformers
from huggingface_hub import model_info
from lm_eval.tasks import TaskManager
from lm_eval.utils import make_table

MODEL_ID = "meta-llama/Llama-2-13b-hf"
TASKS = ["lambada_openai", "arc_easy", "piqa", "copa", "qnli", "sst2"]
NUM_FEWSHOT = 0
BATCH_SIZE = "auto"
MAX_BATCH_SIZE = 64
BOOTSTRAP_ITERS = 0  # Skip expensive bootstrap; accuracy values are unchanged.
LOG_SAMPLES = True
OUTPUT_DIR = Path("result")
OUTPUT_PATH = OUTPUT_DIR / "llama2-13b-fp16-zero-shot.json"
REQUEST_CACHE_PATH = OUTPUT_DIR / "llama2-13b-fp16-zero-shot-logged-cache"

# Full-dataset closure. lm-eval uses the labeled validation split when test labels are unavailable.
EXPECTED_SAMPLES = {
    "lambada_openai": 5153,
    "arc_easy": 2376,
    "piqa": 1838,
    "copa": 100,
    "qnli": 5463,
    "sst2": 872,
}

# Metric used in the BiE-style table. Every raw metric is still retained in the JSON.
PAPER_METRICS = {
    "lambada_openai": "acc,none",
    "arc_easy": "acc_norm,none",
    "piqa": "acc_norm,none",
    "copa": "acc,none",
    "qnli": "acc,none",
    "sst2": "acc,none",
}
DISPLAY_NAMES = {
    "lambada_openai": "LAMBADA",
    "arc_easy": "ARC-Easy",
    "piqa": "PIQA",
    "copa": "COPA",
    "qnli": "QNLI",
    "sst2": "SST-2",
}

if not torch.cuda.is_available():
    raise RuntimeError("This baseline requires an NVIDIA CUDA GPU.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(1234)
torch.backends.cuda.matmul.allow_tf32 = False

## Validate the protocol

Before downloading the model, verify that all six pinned task names exist in this lm-eval release. `sst2` is lm-eval 0.4.13's task name for the SST-2 dataset.

In [ ]:
task_manager = TaskManager()
matched_tasks = task_manager.match_tasks(TASKS)
missing_tasks = sorted(set(TASKS) - set(matched_tasks))
if missing_tasks:
    raise RuntimeError(f"Tasks missing from lm-eval: {missing_tasks}")

print(f"lm-eval: {importlib.metadata.version('lm_eval')}")
print(f"Tasks: {', '.join(TASKS)}")
print(f"Expected examples: {sum(EXPECTED_SAMPLES.values()):,}")

## Run the FP16 zero-shot evaluation

Accept the LLaMA-2 license on Hugging Face and either run `huggingface-cli login` or set `HF_TOKEN`. The Hugging Face backend loads the unmodified model in FP16 on GPU 0. `batch_size=auto` only selects an efficient batch size; it does not change the scoring protocol.

In [ ]:
hf_token = os.getenv("HF_TOKEN")
resolved_model_revision = model_info(MODEL_ID, token=hf_token).sha

raw_results = lm_eval.simple_evaluate(
    model="hf",
    model_args=(
        f"pretrained={MODEL_ID},"
        "dtype=float16,"
        "attn_implementation=eager,"
        f"revision={resolved_model_revision}"
    ),
    tasks=TASKS,
    num_fewshot=NUM_FEWSHOT,
    batch_size=BATCH_SIZE,
    max_batch_size=MAX_BATCH_SIZE,
    device="cuda:0",
    limit=None,
    use_cache=str(REQUEST_CACHE_PATH),
    bootstrap_iters=BOOTSTRAP_ITERS,
    check_integrity=False,
    log_samples=LOG_SAMPLES,
    task_manager=task_manager,
    random_seed=0,
    numpy_random_seed=1234,
    torch_random_seed=1234,
    fewshot_random_seed=1234,
)

if raw_results is None:
    raise RuntimeError("lm-eval returned no results.")

print(make_table(raw_results))

## Evaluation closure and result export

The run is accepted only if every task reports the expected number of evaluated examples. The summary percentages reproduce the columns used by BiE, while `lm_eval_raw` preserves all metrics, task versions, task configs, and effective sample counts for future analysis. Bootstrap standard-error estimation is disabled because it does not change accuracy and would add avoidable runtime.

In [ ]:
def to_jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def metric_value(task_name, metric_name):
    metrics = raw_results["results"][task_name]
    if metric_name not in metrics:
        raise KeyError(
            f"Missing {metric_name!r} for {task_name}; available: {sorted(metrics)}"
        )
    return float(metrics[metric_name])


closure = {}
for task_name, expected in EXPECTED_SAMPLES.items():
    sample_info = raw_results["n-samples"][task_name]
    original = int(sample_info["original"])
    effective = int(sample_info["effective"])
    passed = original == expected and effective == expected
    closure[task_name] = {
        "expected": expected,
        "original": original,
        "effective": effective,
        "passed": passed,
    }
    if not passed:
        raise RuntimeError(
            f"Evaluation closure failed for {task_name}: "
            f"expected={expected}, original={original}, effective={effective}"
        )

paper_scores = {}
for task_name in TASKS:
    metric_name = PAPER_METRICS[task_name]
    score = metric_value(task_name, metric_name)
    stderr_name = metric_name.replace(",none", "_stderr,none")
    stderr_raw = raw_results["results"][task_name].get(stderr_name)
    try:
        stderr = float(stderr_raw)
    except (TypeError, ValueError):
        stderr = None
    paper_scores[task_name] = {
        "display_name": DISPLAY_NAMES[task_name],
        "metric": metric_name,
        "score": score,
        "score_percent": score * 100.0,
        "stderr": stderr,
        "stderr_percent": None if stderr is None else stderr * 100.0,
    }

macro_average = sum(x["score"] for x in paper_scores.values()) / len(paper_scores)
raw_jsonable = json.loads(
    json.dumps(raw_results, ensure_ascii=False, default=to_jsonable)
)

result = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "fp16_zero_shot_accuracy_baseline",
    "model": MODEL_ID,
    "model_revision": resolved_model_revision,
    "dtype": "float16",
    "quantized": False,
    "attention_implementation": "eager",
    "num_fewshot": NUM_FEWSHOT,
    "limit": None,
    "batch_size": BATCH_SIZE,
    "max_batch_size": MAX_BATCH_SIZE,
    "bootstrap_iters": BOOTSTRAP_ITERS,
    "log_samples": LOG_SAMPLES,
    "paper_metric_policy": {
        "lambada": "acc",
        "arc_easy_and_piqa": "acc_norm",
        "copa_qnli_sst2": "acc",
    },
    "paper_scores": paper_scores,
    "macro_average": macro_average,
    "macro_average_percent": macro_average * 100.0,
    "evaluation_closure": closure,
    "total_evaluated_examples": sum(
        item["effective"] for item in closure.values()
    ),
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "lm_eval": importlib.metadata.version("lm_eval"),
    },
    "lm_eval_raw": raw_jsonable,
}

OUTPUT_PATH.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nBiE-compatible FP16 baseline")
for task_name, item in paper_scores.items():
    print(f"{item['display_name']:>10}: {item['score_percent']:.2f}%")
print(f"{'Average':>10}: {result['macro_average_percent']:.2f}%")
print(f"Closure: {result['total_evaluated_examples']:,} / {sum(EXPECTED_SAMPLES.values()):,}")
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))